# NLP Bot Accuracy Evaluation

This notebook evaluates the accuracy of our AI-powered NLP function (`matcher.py`) in extracting the correct IFC element GUIDs from natural language text.

## Evaluation Workflow:
1. Load human-annotated test cases (input text + expected GUIDs)
2. Run NLP matcher on each test case
3. Compare returned GUIDs vs expected GUIDs
4. Calculate accuracy metrics (Exact Match, Precision, Recall, F1 Score)
5. Display summary statistics and per-test-case breakdown

## 1. Import Required Libraries

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Add parent directory to path to import NLP module
sys.path.insert(0, str(Path.cwd().parent))

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Test Data and Import NLP Matcher

In [2]:
# Load test data
test_data_path = Path("test_data.json")
with open(test_data_path, 'r', encoding='utf-8') as f:
    test_cases = json.load(f)

print(f"Loaded {len(test_cases)} test cases")
print("\nSample test case:")
print(json.dumps(test_cases[0], indent=2))

Loaded 33 test cases

Sample test case:
{
  "test_id": 1,
  "difficulty": "easy",
  "category": "door",
  "input_text": "[UPDATE]\nDoor Single Flush Inside Level 1\nInstalled",
  "expected_guids": [
    "2cXV28XOjE6f6irgi0COIY",
    "2cXV28XOjE6f6irgi0COIZ",
    "2cXV28XOjE6f6irgi0COI_",
    "2cXV28XOjE6f6irgi0COIv",
    "2cXV28XOjE6f6irgi0COIw",
    "2cXV28XOjE6f6irgi0COIx",
    "2cXV28XOjE6f6irgi0COIy",
    "2cXV28XOjE6f6irgi0COIz",
    "2cXV28XOjE6f6irgi0COgC",
    "2cXV28XOjE6f6irgi0COgH",
    "2cXV28XOjE6f6irgi0COhu",
    "2cXV28XOjE6f6irgi0COmZ",
    "2cXV28XOjE6f6irgi0COok",
    "2cXV28XOjE6f6irgi0COp7",
    "2cXV28XOjE6f6irgi0COru",
    "2cXV28XOjE6f6irgi0COtb"
  ],
  "description": "All 16 single-flush inside doors on Level 1"
}


In [3]:
# Import the NLP matcher function
from NLP.io_wrapper import run_sample_match  # Adjust function name as needed

print("NLP matcher imported successfully!")

NLP matcher imported successfully!


## 3. Run NLP Matcher on Test Cases

In [4]:
results = []

for test_case in test_cases:
    test_id = test_case['test_id']
    input_text = test_case['input_text']
    expected_guids = set(test_case['expected_guids'])
    difficulty = test_case.get('difficulty', 'unknown')  # Get difficulty level
    
    # Run the NLP matcher
    try:
        # Call run_sample_match with the input text (as used in bot_logger.py)
        match_result = run_sample_match(input_text)
        
        # Extract guid from the result dictionary
        # guid can be a single string or a list of strings
        guid_data = match_result.get("guid")
        if guid_data:
            if isinstance(guid_data, list):
                returned_guids = set(guid_data)
            else:
                returned_guids = set([guid_data])
        else:
            returned_guids = set()
        
        # Check for any error message
        error = match_result.get("error")
    except Exception as e:
        returned_guids = set()
        error = str(e)
    
    # Determine exact match
    exact_match = 1 if expected_guids == returned_guids else 0
    
    # Store only essential fields - metrics calculated in summary cells
    results.append({
        'test_id': test_id,
        'difficulty': difficulty,
        'input_text': input_text,
        'exact_match': exact_match,
        'expected_guids': expected_guids,  # Keep as set for easier comparison
        'returned_guids': returned_guids,  # Keep as set for easier comparison
        'error': error
    })

# Convert to DataFrame
df_results = pd.DataFrame(results)
print(f"Completed evaluation of {len(results)} test cases")

Completed evaluation of 33 test cases


## 4. Summary Accuracy Metrics

In [5]:
# Calculate overall metrics from the stored GUID sets
total_tests = len(df_results)
exact_match_rate = df_results['exact_match'].mean() * 100

# Calculate precision, recall, and F1 from GUID sets
precisions = []
recalls = []
f1_scores = []

for _, row in df_results.iterrows():
    expected = row['expected_guids']
    returned = row['returned_guids']
    
    tp = len(expected & returned)
    fp = len(returned - expected)
    fn = len(expected - returned)
    
    precision = tp / len(returned) if returned else 0
    recall = tp / len(expected) if expected else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)

avg_precision = sum(precisions) / len(precisions) * 100 if precisions else 0
avg_recall = sum(recalls) / len(recalls) * 100 if recalls else 0
avg_f1 = sum(f1_scores) / len(f1_scores) * 100 if f1_scores else 0

# Create summary table
summary_data = {
    'Metric': ['Exact Match Rate', 'Average Precision', 'Average Recall', 'Average F1 Score'],
    'Percentage': [
        f"{exact_match_rate:.2f}%",
        f"{avg_precision:.2f}%",
        f"{avg_recall:.2f}%",
        f"{avg_f1:.2f}%"
    ],
    'Raw Score': [
        f"{df_results['exact_match'].sum()}/{total_tests}",
        f"{sum(precisions) / len(precisions):.4f}",
        f"{sum(recalls) / len(recalls):.4f}",
        f"{sum(f1_scores) / len(f1_scores):.4f}"
    ]
}

df_summary = pd.DataFrame(summary_data)
print("="*60)
print("OVERALL ACCURACY SUMMARY")
print("="*60)
print(df_summary.to_string(index=False))
print("="*60)

OVERALL ACCURACY SUMMARY
           Metric Percentage Raw Score
 Exact Match Rate     54.55%     18/33
Average Precision     54.55%    0.5455
   Average Recall     54.55%    0.5455
 Average F1 Score     54.55%    0.5455


## 4. Results by Difficulty Level

Test cases are classified into three difficulty levels:
- **Easy**: Clear element type + level specification (e.g., "ceiling compound L1", "Floor 150 Level 2")
- **Medium**: Missing one significant token (e.g., no level specified, dimension-only, ambiguous scope)
- **Hard**: Vague or contextual queries requiring inference (e.g., "L2 all elements")

In [6]:
# Separate results by difficulty
df_easy = df_results[df_results['difficulty'] == 'easy'].copy()
df_medium = df_results[df_results['difficulty'] == 'medium'].copy()
df_hard = df_results[df_results['difficulty'] == 'hard'].copy()

# Calculate metrics for each difficulty level
def calculate_difficulty_metrics(df):
    if len(df) == 0:
        return {
            'total_tests': 0,
            'exact_matches': 0,
            'partial_matches': 0,
            'exact_match_rate': 0.0
        }
    
    exact = len(df[df['exact_match'] == 1])
    partial = len(df[df['exact_match'] == 0])
    
    return {
        'total_tests': len(df),
        'exact_matches': exact,
        'partial_matches': partial,
        'exact_match_rate': (exact / len(df)) * 100 if len(df) > 0 else 0.0
    }

easy_metrics = calculate_difficulty_metrics(df_easy)
medium_metrics = calculate_difficulty_metrics(df_medium)
hard_metrics = calculate_difficulty_metrics(df_hard)

# Display comparison table
difficulty_comparison = pd.DataFrame({
    'Difficulty': ['Easy', 'Medium', 'Hard'],
    'Total Tests': [easy_metrics['total_tests'], medium_metrics['total_tests'], hard_metrics['total_tests']],
    'Exact Matches': [easy_metrics['exact_matches'], medium_metrics['exact_matches'], hard_metrics['exact_matches']],
    'Partial Matches': [easy_metrics['partial_matches'], medium_metrics['partial_matches'], hard_metrics['partial_matches']],
    'Accuracy %': [f"{easy_metrics['exact_match_rate']:.1f}%", 
                   f"{medium_metrics['exact_match_rate']:.1f}%", 
                   f"{hard_metrics['exact_match_rate']:.1f}%"]
})

print("\n" + "="*80)
print("ACCURACY BY DIFFICULTY LEVEL")
print("="*80)
print(difficulty_comparison.to_string(index=False))
print("="*80)


ACCURACY BY DIFFICULTY LEVEL
Difficulty  Total Tests  Exact Matches  Partial Matches Accuracy %
      Easy           20             16                4      80.0%
    Medium            8              2                6      25.0%
      Hard            5              0                5       0.0%


### 4a. Easy Test Results

In [7]:
# Display Easy test results
if len(df_easy) > 0:
    # Create display dataframe for easy tests
    easy_display = []
    for _, row in df_easy.iterrows():
        easy_display.append({
            'Test': row['test_id'],
            'Match': '✓' if row['exact_match'] == 1 else '✗',
            'Input Text': row['input_text'][:50] + '...' if len(row['input_text']) > 50 else row['input_text'],
            'Expected': f"{len(row['expected_guids'])} GUIDs",
            'Returned': f"{len(row['returned_guids'])} GUIDs" if row['returned_guids'] else "0 GUIDs",
            'Status': row.get('status', ''),
            'Error': row.get('error', '')
        })
    
    df_easy_display = pd.DataFrame(easy_display)
    print(df_easy_display.to_string(index=False))
    
    # Show failures in detail
    easy_failures = df_easy[df_easy['exact_match'] == 0]
    if len(easy_failures) > 0:
        print("\n" + "="*80)
        print("EASY TEST FAILURES (DETAILED)")
        print("="*80)
        for _, row in easy_failures.iterrows():
            print(f"\nTest {row['test_id']}:")
            print(f"  Input: {row['input_text']}")
            print(f"  Expected ({len(row['expected_guids'])} GUIDs): {sorted(row['expected_guids'])}")
            print(f"  Returned ({len(row['returned_guids']) if row['returned_guids'] else 0} GUIDs): {sorted(row['returned_guids']) if row['returned_guids'] else []}")
            
            if row['expected_guids'] and row['returned_guids']:
                missing = row['expected_guids'] - row['returned_guids']
                extra = row['returned_guids'] - row['expected_guids']
                if missing:
                    print(f"  ❌ Missing: {sorted(missing)}")
                if extra:
                    print(f"  ➕ Extra: {sorted(extra)}")
            
            if row.get('error'):
                print(f"  Error: {row['error']}")
else:
    print("No easy tests found.")

 Test Match                                              Input Text Expected Returned Status                                                                                                                                                                                                                                         Error
    1     ✓ [UPDATE]\nDoor Single Flush Inside Level 1\nInstalle... 16 GUIDs 16 GUIDs                                                                                                                                 Matched 16 element(s) via token/level filters. Level filter applied: level 1. Status refined using allowed statuses.
    2     ✓            [UPDATE] Door Double Glass Wood L1 Specified  2 GUIDs  2 GUIDs                                                                                                                                  Matched 2 element(s) via token/level filters. Level filter applied: level 1. Status refined using allowed statuses.
    5  

### 4b. Medium Test Results

In [8]:
# Display Medium test results
if len(df_medium) > 0:
    # Create display dataframe for medium tests
    medium_display = []
    for _, row in df_medium.iterrows():
        medium_display.append({
            'Test': row['test_id'],
            'Match': '✓' if row['exact_match'] == 1 else '✗',
            'Input Text': row['input_text'][:50] + '...' if len(row['input_text']) > 50 else row['input_text'],
            'Expected': f"{len(row['expected_guids'])} GUIDs",
            'Returned': f"{len(row['returned_guids'])} GUIDs" if row['returned_guids'] else "0 GUIDs",
            'Status': row.get('status', ''),
            'Error': row.get('error', '')
        })
    
    df_medium_display = pd.DataFrame(medium_display)
    print(df_medium_display.to_string(index=False))
    
    # Show failures in detail
    medium_failures = df_medium[df_medium['exact_match'] == 0]
    if len(medium_failures) > 0:
        print("\n" + "="*80)
        print("MEDIUM TEST FAILURES (DETAILED)")
        print("="*80)
        for _, row in medium_failures.iterrows():
            print(f"\nTest {row['test_id']}:")
            print(f"  Input: {row['input_text']}")
            print(f"  Expected ({len(row['expected_guids'])} GUIDs): {sorted(row['expected_guids'])}")
            print(f"  Returned ({len(row['returned_guids']) if row['returned_guids'] else 0} GUIDs): {sorted(row['returned_guids']) if row['returned_guids'] else []}")
            
            if row['expected_guids'] and row['returned_guids']:
                missing = row['expected_guids'] - row['returned_guids']
                extra = row['returned_guids'] - row['expected_guids']
                if missing:
                    print(f"  ❌ Missing: {sorted(missing)}")
                if extra:
                    print(f"  ➕ Extra: {sorted(extra)}")
            
            if row.get('error'):
                print(f"  Error: {row['error']}")
else:
    print("No medium tests found.")

 Test Match                               Input Text Expected Returned Status                                                                                                                                                                                                                        Error
    3     ✗      [UPDATE]\ndors inside L1\nInspected 16 GUIDs  0 GUIDs                                                                     No confident GUID match; leaving status as manual review. Model rationale: The update mentions 'Inspected' but does not clearly match any specific window GUID.
    8     ✗              [UPDATE]\nrof L2\nInspected  1 GUIDs  0 GUIDs                                   No confident GUID match; leaving status as manual review. Model rationale: The update mentions 'Inspected' but does not clearly match any specific element related to the roof or its properties.
   10     ✗     [UPDATE] roof 400 level 2 acceptance  1 GUIDs  0 GUIDs                                 

### 4c. Hard Test Results

In [9]:
# Display Hard test results
if len(df_hard) > 0:
    # Create display dataframe for hard tests
    hard_display = []
    for _, row in df_hard.iterrows():
        hard_display.append({
            'Test': row['test_id'],
            'Match': '✓' if row['exact_match'] == 1 else '✗',
            'Input Text': row['input_text'][:50] + '...' if len(row['input_text']) > 50 else row['input_text'],
            'Expected': f"{len(row['expected_guids'])} GUIDs",
            'Returned': f"{len(row['returned_guids'])} GUIDs" if row['returned_guids'] else "0 GUIDs",
            'Status': row.get('status', ''),
            'Error': row.get('error', '')
        })
    
    df_hard_display = pd.DataFrame(hard_display)
    print(df_hard_display.to_string(index=False))
    
    # Show failures in detail
    hard_failures = df_hard[df_hard['exact_match'] == 0]
    if len(hard_failures) > 0:
        print("\n" + "="*80)
        print("HARD TEST FAILURES (DETAILED)")
        print("="*80)
        for _, row in hard_failures.iterrows():
            print(f"\nTest {row['test_id']}:")
            print(f"  Input: {row['input_text']}")
            print(f"  Expected ({len(row['expected_guids'])} GUIDs): {sorted(row['expected_guids'])}")
            print(f"  Returned ({len(row['returned_guids']) if row['returned_guids'] else 0} GUIDs): {sorted(row['returned_guids']) if row['returned_guids'] else []}")
            
            if row['expected_guids'] and row['returned_guids']:
                missing = row['expected_guids'] - row['returned_guids']
                extra = row['returned_guids'] - row['expected_guids']
                if missing:
                    print(f"  ❌ Missing: {sorted(missing)}")
                if extra:
                    print(f"  ➕ Extra: {sorted(extra)}")
            
            if row.get('error'):
                print(f"  Error: {row['error']}")
else:
    print("No hard tests found.")

 Test Match                           Input Text Expected Returned Status                                                                                                                                                                                                   Error
    4     ✗   [UPDATE] all dors lvl 1 acceptance 18 GUIDs  0 GUIDs              No confident GUID match; leaving status as manual review. Model rationale: The update mentions acceptance of all doors on level 1, but does not specify which doors or provide clear identifiers.
    9     ✗      [UPDATE] rof generic acceptance  1 GUIDs  0 GUIDs                 No confident GUID match; leaving status as manual review. Model rationale: The update 'rof generic acceptance' does not clearly match any specific BIM/IFC element based on the provided data.
   14     ✗ [UPDATE] flors all levels acceptance  3 GUIDs  0 GUIDs                                      No confident GUID match; leaving status as manual review. Model rationale:

## 5. Detailed Per-Test-Case Results

In [10]:
# Display 1:1 matching comparison in tabular form
print("\nDETAILED TEST RESULTS - Expected vs Returned GUIDs:")
print("="*150)

# Create a simplified display dataframe
display_data = []
for idx, row in df_results.iterrows():
    match_symbol = '✓' if row['exact_match'] == 1 else '✗'
    
    # Calculate counts from GUID sets
    expected_count = len(row['expected_guids'])
    returned_count = len(row['returned_guids'])
    
    # Format GUID lists for display
    expected_list = list(row['expected_guids'])[:2]
    returned_list = list(row['returned_guids'])[:2] if row['returned_guids'] else []
    expected_str = ', '.join(expected_list) + (f'... (+{expected_count-2})' if expected_count > 2 else '')
    returned_str = ', '.join(returned_list) + (f'... (+{returned_count-2})' if returned_count > 2 else '')
    
    # Calculate match info
    if row['exact_match'] != 1:
        missing = len(row['expected_guids'] - row['returned_guids']) if row['returned_guids'] else expected_count
        extra = len(row['returned_guids'] - row['expected_guids']) if row['returned_guids'] else 0
        match_info = f"Missing: {missing}, Extra: {extra}" if (missing or extra) else "Match"
    else:
        match_info = "Perfect Match"
    
    display_data.append({
        'Test': row['test_id'],
        'Match': match_symbol,
        'Input Text': row['input_text'][:50] + '...' if len(row['input_text']) > 50 else row['input_text'],
        'Expected': f"{expected_count} GUIDs",
        'Returned': f"{returned_count} GUIDs",
        'Status': match_info,
        'Error': row['error'] if row['error'] else ''
    })

df_display = pd.DataFrame(display_data)
print(df_display.to_string(index=False))
print("="*150)

# Show full details for failed tests
failed_tests = df_results[df_results['exact_match'] != 1]
if not failed_tests.empty:
    print(f"\n\n🔍 DETAILED VIEW OF {len(failed_tests)} FAILED TESTS:")
    print("="*150)
    for idx, row in failed_tests.iterrows():
        expected_count = len(row['expected_guids'])
        returned_count = len(row['returned_guids'])
        
        print(f"\n📌 Test {row['test_id']}: {row['input_text']}")
        print(f"   Expected ({expected_count}): {sorted(row['expected_guids'])}")
        print(f"   Returned ({returned_count}): {sorted(row['returned_guids']) if row['returned_guids'] else []}")
        
        missing = row['expected_guids'] - row['returned_guids']
        extra = row['returned_guids'] - row['expected_guids']
        if missing:
            print(f"   ❌ Missing: {sorted(missing)}")
        if extra:
            print(f"   ➕ Extra: {sorted(extra)}")
    print("="*150)


DETAILED TEST RESULTS - Expected vs Returned GUIDs:
 Test Match                                              Input Text Expected Returned                Status                                                                                                                                                                                                                                         Error
    1     ✓ [UPDATE]\nDoor Single Flush Inside Level 1\nInstalle... 16 GUIDs 16 GUIDs         Perfect Match                                                                                                                          Matched 16 element(s) via token/level filters. Level filter applied: level 1. Status refined using allowed statuses.
    2     ✓            [UPDATE] Door Double Glass Wood L1 Specified  2 GUIDs  2 GUIDs         Perfect Match                                                                                                                           Matched 2 element(s) via 

## 8. Export Results to CSV

In [ ]:
# Export the same tabular format to CSV
export_data = []
for idx, row in df_results.iterrows():
    match_symbol = '✓' if row['exact_match'] == 1 else '✗'
    
    # Calculate counts and match info from GUID sets
    expected_count = len(row['expected_guids'])
    returned_count = len(row['returned_guids'])
    
    if row['exact_match'] != 1:
        missing = len(row['expected_guids'] - row['returned_guids']) if row['returned_guids'] else expected_count
        extra = len(row['returned_guids'] - row['expected_guids']) if row['returned_guids'] else 0
        match_info = f"Missing: {missing}, Extra: {extra}" if (missing or extra) else "Match"
    else:
        match_info = "Perfect Match"
    
    export_data.append({
        'Test_ID': row['test_id'],
        'Difficulty': row['difficulty'],  # Add difficulty column
        'Match': match_symbol,
        'Input_Text': row['input_text'],
        'Expected_Count': expected_count,
        'Returned_Count': returned_count,
        'Status': match_info,
        'Expected_GUIDs': ', '.join(sorted(row['expected_guids'])),
        'Returned_GUIDs': ', '.join(sorted(row['returned_guids'])) if row['returned_guids'] else '',
        'Error': row['error'] if row['error'] else ''
    })

df_export = pd.DataFrame(export_data)
output_path = Path("nlp_evaluation_results.csv")
df_export.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n✓ Results exported to: {output_path.absolute()}")

# Also save summary metrics
summary_output = Path("nlp_evaluation_summary.txt")
with open(summary_output, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("NLP BOT ACCURACY EVALUATION SUMMARY\n")
    f.write("="*60 + "\n\n")
    f.write(f"Total Test Cases: {total_tests}\n")
    f.write(f"Exact Match Rate: {exact_match_rate:.2f}%\n")
    f.write(f"Average Precision: {avg_precision:.2f}%\n")
    f.write(f"Average Recall: {avg_recall:.2f}%\n")
    f.write(f"Average F1 Score: {avg_f1:.2f}%\n\n")
    
    # Add difficulty breakdown
    f.write("="*60 + "\n")
    f.write("ACCURACY BY DIFFICULTY LEVEL\n")
    f.write("="*60 + "\n\n")


✓ Results exported to: c:\Users\User\Documents\GitHub\spatial_acc_telebot\tests\nlp_evaluation_results.csv


: 